# 02 - Figure 2: transport phenotype

**Question.** What statistical transport phenotype distinguishes the
team centroid from its constituent player trajectories?

| Panel | Analysis |
|---|---|
| A | Run-duration survivor functions |
| B | Run-length survivor functions |
| C | Mean-squared displacement and fitted scaling ranges |
| D | Player MSD decomposed into centroid translation and motion within the formation |

In [ ]:
from pathlib import Path
import subprocess
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display


# Locate the repository before importing its analysis package. This works when
# Jupyter starts from either the repo root or this notebook directory.
_start = Path.cwd()
ROOT = next(
    path for path in (_start, *_start.parents)
    if (path / "analysis" / "levy_paper").is_dir()
    and (path / "requirements.txt").is_file()
)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from analysis.levy_paper.util.publication_notebook_utils import (
    PRIMARY_CACHE_SUFFIX,
    cache_path as make_cache_path,
    csv_shapes,
    display_live_or_frozen,
    file_status,
    hazard_support_summary,
    load_processed_cache,
    order_state_summary,
    panel_inventory,
    publication_paths,
    relative_path,
    resolve_data_mode,
    table_inventory,
    transition_row_sum_audit,
    transport_run_summary,
)

PATHS = publication_paths(ROOT)
LEVY_DIR = PATHS["levy_dir"]
DATA_DIR = PATHS["data_dir"]
PRIMARY_CACHE_DIR = PATHS["primary_cache_dir"]
FINAL_FIGURES = PATHS["final_figures"]
SOURCE_DATA = PATHS["source_data"]
SUPPLEMENT = PATHS["supplement"]
CACHE_SUFFIX = PRIMARY_CACHE_SUFFIX

# DATA_MODE options:
#   "auto"     use processed caches when all required files exist;
#              otherwise use tracked reviewer tables/frozen figures
#   "cache"    require processed caches and fail clearly if they are missing
#   "reviewer" use only tracked public artefacts
DATA_MODE = "auto"
BUILD_FIGURE = True
SAVE_FIGURE_OUTPUTS = True
DISPLAY_FROZEN_OUTPUT = True
REBUILD_CACHE_FROM_AWS = False
REFIT_FIGURE4_FIGURE5_MODELS = False
USE_VERSIONED_FINAL_FIGURE4_FIT = True


def rel(path):
    return relative_path(path, ROOT)


def show_file_status(paths):
    return file_status(paths, ROOT)


def show_csv_shapes(paths):
    return csv_shapes(paths, ROOT)


def cache_path(stem):
    return make_cache_path(PRIMARY_CACHE_DIR, stem, CACHE_SUFFIX)


def show_figure(fig, frozen_path, width=1100):
    return display_live_or_frozen(
        fig,
        frozen_path,
        display_frozen=DISPLAY_FROZEN_OUTPUT,
        width=width,
    )

## 1. Load processed transport data

In [ ]:
from analysis.levy_paper.scripts import create_figure2_transport_phenotype as figure2_base
from analysis.levy_paper.scripts import create_final_figure2_transport_phenotype as figure2

figure2_inputs = {
    "runs": cache_path("runs_long"),
    "msd": cache_path("msd_long"),
    "trajectory": cache_path("trajectory_long"),
}
resolved_mode = resolve_data_mode(DATA_MODE, list(figure2_inputs.values()))
cache_ready = resolved_mode == "cache"
print("resolved_data_mode", resolved_mode)
display(show_file_status(figure2_inputs.values()))

inputs = {}
if cache_ready:
    inputs = {name: pd.read_parquet(path) for name, path in figure2_inputs.items()}
    display(table_inventory(inputs))
    display(transport_run_summary(inputs["runs"]))

## 2. Compute the common panel context

In [ ]:
context = None
if inputs:
    context = figure2_base.prepare_figure2_context(inputs=inputs, write_audits=False)
    context["completed_centroid_runs"] = figure2.completed_centroid_runs(inputs["runs"])
    display(panel_inventory({
        "duration survivor audit": context["duration_audit"],
        "length survivor audit": context["length_audit"],
        "MSD aggregate": context["msd_agg"],
        "MSD bootstrap": context["msd_boot"],
        "MSD decomposition": context["decomp"],
    }))

## 3. Panels A and B - run survivor functions

In [ ]:
if context is not None:
    survivor_summary = pd.DataFrame([
        {
            "population": "centroid",
            "runs": len(context["centroid_runs"]),
            "median_duration_s": context["centroid_runs"]["duration_s"].median(),
            "median_length_m": context["centroid_runs"]["run_length_m"].median(),
        },
        {
            "population": "player",
            "runs": len(context["player_runs"]),
            "median_duration_s": context["player_runs"]["duration_s"].median(),
            "median_length_m": context["player_runs"]["run_length_m"].median(),
        },
    ])
    display(survivor_summary)
    display(context["duration_audit"].head(10))
    display(context["length_audit"].head(10))

## 4. Panel C - MSD scaling

In [ ]:
if context is not None:
    display(context["scaling_audit"])
    display(context["local_slope_audit"].head(12))

## 5. Panel D - centroid-frame MSD decomposition

In [ ]:
if context is not None:
    display(context["decomp"].head(12))
    display(context["decomp_components"].head(12))

## 6. Duration-tail model used in Panel A

In [ ]:
tail_decision = None
if context is not None:
    tail_audit = figure2.load_tail_audit()
    tail_decision = figure2.broad_tail_decision(tail_audit)
    display(pd.DataFrame([tail_decision]))

## 7. Construct the publication figure

In [ ]:
fig = None
source_tables = None
if BUILD_FIGURE and context is not None:
    fig, source_tables = figure2.plot_final_figure(context, tail_decision)
    if SAVE_FIGURE_OUTPUTS:
        figure2.save_figure_bundle(fig)
        figure2.write_source_data(source_tables)
elif BUILD_FIGURE:
    print("Processed run/MSD/trajectory cache unavailable; using frozen Figure 2.")

display_mode = show_figure(fig, FINAL_FIGURES / "figure2_transport_phenotype.png")
print("figure_display_mode", display_mode)